# 02 — Model training, tuning, and comparison

Run the repository training pipeline first (`python ../train_models.py` from the repository root). This notebook independently documents model behavior and displays saved, real metrics.

**Leakage note:** FinalGrade is deterministically binned from ExamScore. Results with ExamScore are leakage-inflated; without it, scores should be interpreted from the actual measured experiment. The project brief expects roughly 25% chance-level accuracy, but the supplied CSV must be checked if measured accuracy differs.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree
from sklearn.metrics import ConfusionMatrixDisplay
ROOT = Path("..")
comparison_path = ROOT / "models/model_comparison.json"
metrics_path = ROOT / "models/metrics.json"
if not comparison_path.exists():
    raise FileNotFoundError("Run python train_models.py from the project root first.")
comparison = json.loads(comparison_path.read_text())
metrics = json.loads(metrics_path.read_text())
print("Default metrics:", metrics)

## Model results: regression and classification


In [ ]:
reg = pd.DataFrame(comparison["with_exam_score"]["regression"])
clf = pd.DataFrame(comparison["with_exam_score"]["classification"])
display(reg.sort_values("r2",ascending=False))
display(clf.sort_values("macro_f1",ascending=False))

## Linear Regression — baseline regression model

Linear Regression fits a linear relationship. It is the simplest baseline and has no tuning grid.


In [ ]:
display(reg[reg.model=="linear_regression"][["model","r2","mae","rmse"]])

## Decision Tree — regression and classification

The tree learns human-readable if/then splits. The training pipeline tunes depth and minimum leaf size. The plot below shows a shallow view when the selected best tree artifact is a tree; otherwise, it displays a note.


In [ ]:
from joblib import load
try:
    tree_pipe=load(ROOT / "models/regression_decision_tree.pkl")
    plot_tree(tree_pipe.named_steps["model"], max_depth=3, filled=False, feature_names=tree_pipe.named_steps["preprocessor"].get_feature_names_out())
    plt.title("Decision Tree regressor (max depth shown = 3)"); plt.show()
except Exception as e: print("Train artifacts first; tree plot unavailable:", e)
display(reg[reg.model=="decision_tree"][["model","r2","mae","rmse","best_params"]])
display(clf[clf.model=="decision_tree"][["model","accuracy","macro_f1","best_params"]])

## SVM / SVR — RBF kernel and tuning

SVR handles regression; SVC handles classification. Both are sensitive to scale and use the scaled pipeline.


In [ ]:
display(reg[reg.model=="svm"][["model","r2","mae","rmse","best_params"]])
display(clf[clf.model=="svm"][["model","accuracy","macro_f1","best_params"]])

## Random Forest — feature importance

Random Forest averages many constrained trees. The maximum depth and minimum leaf size limit model size.


In [ ]:
display(reg[reg.model=="random_forest"][["model","r2","best_params"]])
display(clf[clf.model=="random_forest"][["model","accuracy","macro_f1","best_params"]])
try:
    rf=load(ROOT / "models/regression_random_forest.pkl")
    importances=rf.named_steps["model"].feature_importances_
    names=rf.named_steps["preprocessor"].get_feature_names_out()
    order=np.argsort(importances)[-15:]
    plt.barh(np.array(names)[order],importances[order]); plt.title("Random Forest feature importance"); plt.show()
except Exception as e: print("Feature importance unavailable:",e)

## Gaussian Naive Bayes — classification only

GaussianNB is fast and compact but assumes each feature contributes independently within a class.


In [ ]:
display(clf[clf.model=="naive_bayes"][["model","accuracy","macro_f1","confusion_matrix"]])
if not clf[clf.model=="naive_bayes"].empty:
    cm=np.array(clf[clf.model=="naive_bayes"].iloc[0]["confusion_matrix"])
    ConfusionMatrixDisplay(cm,display_labels=["High","Good","Average","Low"]).plot(); plt.title("Naive Bayes confusion matrix"); plt.show()

## Model comparison charts


In [ ]:
plt.figure(figsize=(9,4)); plt.bar(reg.model,reg.r2); plt.xticks(rotation=35,ha="right"); plt.ylabel("R²"); plt.title("Regression model comparison (ExamScore included)"); plt.tight_layout(); plt.show()
plt.figure(figsize=(9,4)); plt.bar(clf.model,clf.accuracy); plt.xticks(rotation=35,ha="right"); plt.ylabel("Accuracy"); plt.title("Classification model comparison (ExamScore included)"); plt.tight_layout(); plt.show()

## With versus without ExamScore


In [ ]:
no_reg=pd.DataFrame(comparison["without_exam_score"]["regression"])
no_clf=pd.DataFrame(comparison["without_exam_score"]["classification"])
print("LEAKAGE NOTE:",comparison["leakage_note"])
display(reg[["model","r2","mae","rmse"]].merge(no_reg[["model","r2","mae","rmse"]],on="model",suffixes=("_with_exam","_without_exam")))
display(clf[["model","accuracy","macro_f1"]].merge(no_clf[["model","accuracy","macro_f1"]],on="model",suffixes=("_with_exam","_without_exam")))

## Actual versus predicted and best-classifier confusion matrix


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
try:
    raw=pd.read_csv(ROOT / "data/raw/student_performance.csv").drop_duplicates()
    cols=[c for c in raw.columns if c!="FinalGrade"]
    _,test=train_test_split(raw,test_size=.2,random_state=42,stratify=raw["FinalGrade"])
    X=test[cols]; y=test["FinalGrade"]
    best_reg=load(ROOT / "models/model_v1.pkl"); pred=best_reg.predict(X)
    plt.scatter(y,pred,alpha=.3); plt.xlabel("Actual FinalGrade"); plt.ylabel("Predicted FinalGrade"); plt.title("Actual versus predicted (best regressor)"); plt.show()
    best_clf=load(ROOT / "models/best_classifier.pkl"); cp=best_clf.predict(X)
    from sklearn.metrics import confusion_matrix
    ConfusionMatrixDisplay(confusion_matrix(y,cp,labels=[0,1,2,3]),display_labels=["High","Good","Average","Low"]).plot(); plt.title("Best classifier confusion matrix"); plt.show()
except Exception as e: print("Plots require current trained artifacts:",e)

## Conclusion

The best model names and real scores are selected and written by `model_trainer.py`. Compare R² for regression and macro-F1 for classification. With ExamScore, the winner is expected to exploit the deterministic target bands; without ExamScore, all models face the genuine, much harder task of inferring grades from other attributes. Do not report leakage-inflated scores as independent predictive ability.
